# Scraping a news website for headlines

This code scrapes information from Le Monde's website (https://www.lemonde.fr/en/) to create a csv with each article on the homepages' title, subhead, article URL, whether it's premium or not, byline, article type and image URL. 

## Importing the necessary packages and data

In [30]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

Getting the homepage data and saving it in a variable called doc

In [31]:
response = requests.get("https://www.lemonde.fr/en/")
doc = BeautifulSoup(response.text, 'html.parser')

## Searching the webpage data for the elements I need

Based on using the inspect tool on the website, I found the element div.article.article--headlines is being used to display several articles on the page. But that doesn't look like it will work for everything as different articles display on the page different. Thus I started by at any element with the article class.

In [32]:
stories = doc.find_all(class_="article")
stories

[<div class="article article--main" data-zone="titles title_1"> <div class="lmd-link-clickarea"> <span class="icon__premium icon--outside" id="zone1-premium-96238"><span class="sr-only">Subscribers only</span></span><h1 class="article__title"><a aria-describedby="zone1-premium-96238" class="lmd-link-clickarea__link" href="https://www.lemonde.fr/en/international/article/2026/06/17/trump-buoyed-by-his-deal-with-iran-shows-renewed-interest-in-ukraine-at-g7-summit_6754581_4.html"> <p class="article__title-label">Trump, buoyed by his deal with Iran, shows renewed interest in Ukraine at G7 summit</p> </a> </h1> <div class="article__media-container"> <picture class="article__media"> <img alt="Donald Trump, Emmanuel Macron and Volodymyr Zelensky at the G7 summit in Evian, June 16, 2026." height="2" loading="eager" sizes="(min-width: 1024px) 421px, 100vw" src="https://img.lemde.fr/2026/06/17/333/0/5000/3333/400/266/75/0/d639939_upload-1-9y1lmm3xjfyc-kzihnioglu-evian-g7-16062026-15.jpg" srcset="

#### First attempt at exploring the data using Soma's example code from class

In [33]:
# Starting off without ANY rows
rows = []

for story in stories:
    print("----")
    # Starting off knowing NONE of the columns of data for this datapoint?
    row = {}

    try:
        row['title'] = story.select_one('h1, h2, h3').text.strip()
    except:
        print("Headline not found!")
        continue
        
    try:
        # Find me a media__link OR a reel_link
        row['href'] = story.select_one('a')['href']
    except:
        print("Couldn't find a link")

    try:
        row['tag'] = story.select_one('.ewmvhB').text.strip()
    except:
        print("Couldn't find a tag!")

    try:
        row['tag'] = story.find(attrs={'data-testid': 'card-description'}).text.strip()
    except:
        print("Couldn't find a summary")

    print(row)
    # When we're done adding info to our row, we're going to add it into our list
    # of rows
    rows.append(row)

----
Couldn't find a tag!
Couldn't find a summary
{'title': 'Trump, buoyed by his deal with Iran, shows renewed interest in Ukraine at G7 summit', 'href': 'https://www.lemonde.fr/en/international/article/2026/06/17/trump-buoyed-by-his-deal-with-iran-shows-renewed-interest-in-ukraine-at-g7-summit_6754581_4.html'}
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!


That did not work at all. So I will start exploring the data on my own.

#### Exploring the first 3 and last 3 stories myself just to get a sense of what elements are in it

In [56]:
for story in stories[:3]:
    print("----")
    print(story)

print(" ----- last 3 ----- ")

for story in stories[(len(stories)-3):]:
    print("----")
    print(story)

----
<div class="article article--main" data-zone="titles title_1"> <div class="lmd-link-clickarea"> <span class="icon__premium icon--outside" id="zone1-premium-96238"><span class="sr-only">Subscribers only</span></span><h1 class="article__title"><a aria-describedby="zone1-premium-96238" class="lmd-link-clickarea__link" href="https://www.lemonde.fr/en/international/article/2026/06/17/trump-buoyed-by-his-deal-with-iran-shows-renewed-interest-in-ukraine-at-g7-summit_6754581_4.html"> <p class="article__title-label">Trump, buoyed by his deal with Iran, shows renewed interest in Ukraine at G7 summit</p> </a> </h1> <div class="article__media-container"> <picture class="article__media"> <img alt="Donald Trump, Emmanuel Macron and Volodymyr Zelensky at the G7 summit in Evian, June 16, 2026." height="2" loading="eager" sizes="(min-width: 1024px) 421px, 100vw" src="https://img.lemde.fr/2026/06/17/333/0/5000/3333/400/266/75/0/d639939_upload-1-9y1lmm3xjfyc-kzihnioglu-evian-g7-16062026-15.jpg" srcs

## Building a scraper to capture the information I need

We are looking for...<br>
headline = article__title class <br>
subtitle = article__desc class (only exists for some headlines) <br>
article url = lmd-link-clickarea__link class <br>
image url = src attribute withing the image tag <br>
bylines = article__byline class (but most aren't shown on the homepage) another option I have seen is article__author-name <br>
whether it is premium or not = sr-only class <br>
article type = article__type class (doesn't exist for all stories) <br>

This code uses the above information in a for loop to build a data set with the information we need. I am printing the information for each step to make sure it is working as planned.

In [47]:
rows = []

for story in stories:
    print("----")
    # Starting off knowing NONE of the columns of data for this datapoint?
    row = {}

    try:
        # This gets the headlines generally but will need to be cleaned up using pandas
        row["headline"] = (story.select_one(".article__title").get_text())
        print(row["headline"])
    except:
        print("Headline not found!")
        continue
        
    try:
        # finding the subtitles
        row["subtitle"] = (story.select_one(".article__desc").get_text())
        print(row["subtitle"])
    except:
        print("Couldn't find a subtitle")

    # The .lmd-link-clickarea__link is not getting the article urls for everything unfortunately
    # so I have added one more attempt to catch stragglers before instructing the loop to give up
    try:
        row["article_url"] = story.select_one(".lmd-link-clickarea__link")["href"]
        print(row["article_url"])
    except:
        try:
            row["article_url"] = story.select_one("a")["href"]
            print(row["article_url"])
        except:
            print("Couldn't find an article url")

    try:
        row["image_url"] = story.select_one("img")["src"]
        print(row["image_url"])
    except:
        print("Couldn't find an image_url")
    
    # Bylines appear under different names and many of the articles do not have bylines at all on the homepage
    try:
        row["byline"] = (story.select_one(".article__byline").get_text())
        print(f"byline {row["byline"]}")
    except:
        try:
            row["byline"] = (story.select_one(".article__author-name").get_text())
            print(f"author name {row["byline"]}")
        except:
            print("Couldn't find a byline")

    try:
        if story.select_one(".sr-only") != None:
            row["paywall_status"] = "subscriber only"
            print(row["paywall_status"])
    except:
        print("Couldn't find subscriber status")

    try:
        row["article_type"] = (story.select_one(".article__type").get_text())
        print(row["article_type"])
    except:
        print("Couldn't find article type")
        
    # # When we're done adding info to our row, we're going to add it into our list    
    print(row)
    rows.append(row)

----
 Trump, buoyed by his deal with Iran, shows renewed interest in Ukraine at G7 summit  
European leaders, who would like the US to reengage in the conflict between Moscow and Kyiv, encouraged the American president's enthusiasm about his memorandum of understanding with Tehran and confirmed their offer of assistance in the Strait of Hormuz.
https://www.lemonde.fr/en/international/article/2026/06/17/trump-buoyed-by-his-deal-with-iran-shows-renewed-interest-in-ukraine-at-g7-summit_6754581_4.html
https://img.lemde.fr/2026/06/17/333/0/5000/3333/400/266/75/0/d639939_upload-1-9y1lmm3xjfyc-kzihnioglu-evian-g7-16062026-15.jpg
Couldn't find a byline
subscriber only
Couldn't find article type
{'headline': ' Trump, buoyed by his deal with Iran, shows renewed interest in Ukraine at G7 summit  ', 'subtitle': "European leaders, who would like the US to reengage in the conflict between Moscow and Kyiv, encouraged the American president's enthusiasm about his memorandum of understanding with Tehra

## Exporting the results

Saving it as a data frame and having a quick glance

In [49]:
df = pd.DataFrame(rows)
df

,headline,subtitle,article_url,image_url,paywall_status,byline,article_type
0,"Trump, buoyed by his deal with Iran, shows re...","European leaders, who would like the US to ree...",https://www.lemonde.fr/en/international/articl...,https://img.lemde.fr/2026/06/17/333/0/5000/333...,subscriber only,NaN,NaN
1,French government urged to reveal hidden asset...,NaN,https://www.lemonde.fr/en/politics/article/202...,https://img.lemde.fr/2025/09/19/0/0/1500/1000/...,subscriber only,NaN,NaN
2,French security replaces Palantir with domesti...,NaN,https://www.lemonde.fr/en/france/article/2026/...,https://img.lemde.fr/2026/06/11/0/0/6000/4000/...,subscriber only,NaN,NaN
3,2026 World Cup: International press hails long...,With outstanding performances in their opening...,https://www.lemonde.fr/en/sports/article/2026/...,https://img.lemde.fr/2026/06/16/0/92/5283/3522...,subscriber only,NaN,NaN
4,'Japan's PM hopes to boost growth through publ...,"The Bank of Japan, the last major central bank...",https://www.lemonde.fr/en/economy/article/2026...,https://img.lemde.fr/2026/06/17/0/0/7782/5188/...,subscriber only,NaN,NaN
5,Louvre museum is 'running out of steam' and st...,"'Despite its imposing majesty, despite the dai...",https://www.lemonde.fr/en/culture/article/2026...,https://img.lemde.fr/2026/06/17/0/0/4897/3265/...,NaN,NaN,NaN
6,"Italian historian Carlo Ginzburg, founding fig...",Through his research on the 16th and 17th-cent...,https://www.lemonde.fr/en/obituaries/article/2...,https://img.lemde.fr/2026/06/17/5/0/3872/2581/...,subscriber only,Anne Dujin,Obituary
7,Death of 11-year-old Lyhanna reignites debate ...,NaN,https://www.lemonde.fr/en/france/article/2026/...,https://img.lemde.fr/2026/05/19/0/0/3678/2452/...,subscriber only,NaN,NaN
8,EU lawmakers approve migration reform allowing...,NaN,https://www.lemonde.fr/en/international/articl...,https://img.lemde.fr/2026/06/17/0/0/8083/5388/...,NaN,NaN,NaN
9,"Lionel Messi scores a hat-trick, ties the all-...",NaN,https://www.lemonde.fr/en/international/articl...,https://img.lemde.fr/2026/06/17/0/55/4141/2761...,NaN,NaN,NaN


Exporting the data as a csv

In [51]:
df.to_csv("le_monde_daily_news.csv")

## Making the CSV auto-updating